In [142]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import seaborn as sns
import numpy as np
from datetime import datetime, date, time, timedelta
import matplotlib.pyplot as plt
import seaborn as sns 
import random

In [128]:
try:
    connection = mysql.connector.connect(
        host="localhost",
        user="root",
        password="venomio",
        database="vsp"
    )
    
    if connection.is_connected():       
        cursor = connection.cursor()

        cursor.execute(
        """ 
        SELECT 
            mpb.match_id,
            mpb.sub_in AS sub_minute,
            p.team_id
        FROM match_player_breakdown mpb
        JOIN players p
            ON mpb.player_id = p.id
        WHERE p.team_id IN (31, 30)
        AND mpb.sub_in IS NOT NULL
        """
       )
        column_names = [desc[0] for desc in cursor.description] 
        teams_df = pd.DataFrame(cursor.fetchall(), columns=column_names)
            
except Error as e:
    print(f"Error while connecting to MySQL: {e}")
    
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()

teams_df.head()

,match_id,sub_minute,team_id
0,7716,62,30
1,7716,73,30
2,7716,79,30
3,7716,73,30
4,7717,70,31


In [129]:
try:
    connection = mysql.connector.connect(
        host="localhost",
        user="root",
        password="venomio",
        database="vsp"
    )
    
    if connection.is_connected():       
        cursor = connection.cursor()

        cursor.execute(
            """
            SELECT 
                mpb.sub_in AS sub_minute
            FROM match_player_breakdown mpb
            WHERE mpb.sub_in IS NOT NULL
            """
            )
        column_names = [desc[0] for desc in cursor.description] 
        minutes_df = pd.DataFrame(cursor.fetchall(), columns=column_names)
            
except Error as e:
    print(f"Error while connecting to MySQL: {e}")
    
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()

minutes_df.head()

,sub_minute
0,82
1,30
2,71
3,73
4,61


In [198]:
def _get_rate(numerator: int, denominator: int) -> float:
    if denominator and denominator > 0 and numerator:
        if numerator >= denominator:
            return 1.0
        else:
            return numerator / denominator
    else:
        return 0.0
    
def _sub_count(raw_str: str) -> float:
    count = 0
    try:
        sub_in_list = eval(raw_str)
        count = len(sub_in_list)
    except:
        pass

    return count

def _status_prob(raw_str: str) -> dict:
    counts = {}
    try:
        counts = ast.literal_eval(raw_str)
    except:
        pass

    counts = {k.title(): v for k, v in counts.items()}
    base = {'Leading': 0, 'Level': 0, 'Trailing': 0}
    base.update(counts)

    smoothed = {k: v + 1 for k, v in base.items()}

    total = sum(smoothed.values())
    return {k: v / total for k, v in smoothed.items()}

def _get_players_data(players_df, all_players) -> dict:
    initial_mapping = {}

    players_dict = {}
    for player_id in all_players:
        if player_id in players_df['id'].values:
            player_row = players_df[players_df['id'] == player_id]
            player_sql_data = player_row.iloc[0].to_dict()
            player_sql_data['minutes_played'] = player_sql_data.get('minutes_played')
            player_sql_data['fouls_committed_rate'] = _get_rate(player_sql_data.get('fouls_committed'), player_sql_data.get('minutes_played'))
            player_sql_data['fouls_drawn_rate'] = _get_rate(player_sql_data.get('fouls_drawn'), player_sql_data.get('minutes_played'))
            player_sql_data['yellow_card_rate'] = _get_rate(player_sql_data.get('yellow_cards'), player_sql_data.get('fouls_committed'))
            player_sql_data['red_card_rate'] = _get_rate(player_sql_data.get('red_cards'), player_sql_data.get('fouls_committed'))
            player_sql_data['sub_in_count'] = _sub_count(player_sql_data.get('sub_in'))
            player_sql_data['sub_out_count'] = _sub_count(player_sql_data.get('sub_out'))
            player_sql_data['in_status_prob'] = _status_prob(player_sql_data.get('in_status'))
            player_sql_data['out_status_prob'] = _status_prob(player_sql_data.get('out_status'))
            player_sql_data['out_status_prob'] = _status_prob(player_sql_data.get('out_status'))
            player_sql_data['fr_xg_off_coef'] = player_sql_data['off_xg_coef'] if pd.notna(player_sql_data['off_xg_coef']) else 0
            player_sql_data['fr_xg_def_coef'] = player_sql_data['def_xg_coef'] if pd.notna(player_sql_data['def_xg_coef']) else 0
            delete_columns = ['id', 'team_id', 'off_xg_coef', 'def_xg_coef', 'fouls_committed', 'fouls_drawn', 'yellow_cards', 'red_cards', 'sub_in', 'sub_out', 'in_status', 'out_status', 'fatigue', 'rhythm']
            player_sql_data = {k: v for k, v in player_sql_data.items() if k not in delete_columns}
            players_dict[player_id] = player_sql_data
        else:
            players_dict[player_id] = {
                'minutes_played': 0,
                'fr_xg_off_coef': 0.0,
                'fr_xg_def_coef': 0.0,
                'off_sh_coef': 0.0,
                'def_sh_coef': 0.0,
                'fouls_committed_rate': 0.0,
                'fouls_drawn_rate': 0.0,
                'yellow_card_rate': 0.0,
                'red_card_rate': 0.0,
                'sub_in_count': 0,
                'sub_out_count': 0,
                'in_status_prob': {'Leading': 0.33, 'Level': 0.33, 'Trailing': 0.33},
                'out_status_prob': {'Leading': 0.33, 'Level': 0.33, 'Trailing': 0.33},
            }

        if player_id in initial_mapping:
            extracted = initial_mapping[player_id]
            players_dict[player_id]['bench'] = extracted.get('bench')
            players_dict[player_id]['on_field'] = extracted.get('on_field')
            players_dict[player_id]['yellow_card'] = extracted.get('yellow_card')
            players_dict[player_id]['red_card'] = extracted.get('red_card')
        else:
            continue

    return players_dict

In [207]:
home_starters = ["Andriy Lunin_13_RM", "Éder Militão_3_RM", "Jude Bellingham_5_RM", "Eduardo Camavinga_6_RM", "Vinicius Júnior_7_RM", "Federico Valverde_8_RM", "Kylian Mbappé_9_RM", "Aurélien Tchouaméni_14_RM", "Lucas Vázquez_17_RM", "Antonio Rüdiger_22_RM", "Ferland Mendy_23_RM"]
home_subs = ["Sergio Mestre_34_RM", "Luka Modrić_10_RM", "Arda Güler_15_RM", "Endrick_16_RM", "Jesús Vallejo_18_RM", "Dani Ceballos_19_RM", "Fran Garcia_20_RM", "Brahim Díaz_21_RM"]
away_players = ["Iñaki Peña_13_B", "Pau Cubarsí_2_B", "Alejandro Balde_3_B", "Iñigo Martínez_5_B", "Pedri_8_B", "Robert Lewandowski_9_B", "Raphinha_11_B", "Fermin López_16_B", "Marc Casado_17_B", "Lamine Yamal_19_B", "Jules Koundé_23_B"]
away_subs = ["Wojciech Szczęsny_25_B", "Diego Kochen_31_B", "Gavi_6_B", "Ansu Fati_10_B", "Pablo Torre_14_B", "Pau Victor_18_B", "Dani Olmo_20_B", "Frenkie de Jong_21_B", "Héctor Fort_32_B", "Gerard Martín_35_B", "Sergi Dominguez_36_B"]

In [132]:
all_home_players = home_starters + home_subs
escaped_players = [player.replace("'", "''") for player in all_home_players]
players_str = ", ".join([f"'{player}'" for player in escaped_players])

try:
    connection = mysql.connector.connect(
        host="localhost",
        user="root",
        password="venomio",
        database="vsp"
    )
    
    if connection.is_connected():       
        cursor = connection.cursor()

        cursor.execute(
            f"""
            SELECT 
                *
            FROM players
            WHERE id IN ({players_str})
            """
            )
        column_names = [desc[0] for desc in cursor.description] 
        home_players_df = pd.DataFrame(cursor.fetchall(), columns=column_names)
            
except Error as e:
    print(f"Error while connecting to MySQL: {e}")
    
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()

home_players_df.head()

,id,team_id,off_xg_coef,def_xg_coef,off_sh_coef,def_sh_coef,fatigue,rhythm,minutes_played,fouls_committed,fouls_drawn,yellow_cards,red_cards,sub_in,sub_out,in_status,out_status
0,Andriy Lunin_13_RM,31,0.001149,0.000649,-0.001393,0.006287,0.000,0.051,4494,1,6,2,0,[],[],"{""level"": 0, ""leading"": 0, ""trailing"": 0}","{""level"": 0, ""leading"": 0, ""trailing"": 0}"
1,Antonio Rüdiger_22_RM,31,0.000932,0.000465,0.001999,0.003615,0.530,1.000,9434,64,33,11,0,"[""54"", ""71"", ""74"", ""75"", ""76"", ""77"", ""81"", ""85...","[""50"", ""72"", ""83""]","{""level"": 1, ""leading"": 10, ""trailing"": 0}","{""level"": 0, ""leading"": 3, ""trailing"": 0}"
2,Arda Güler_15_RM,31,0.000030,0.000071,-0.000433,-0.003987,0.291,0.792,976,17,26,2,0,"[""48"", ""65"", ""79"", ""80"", ""81"", ""85"", ""91""]","[""56"", ""66"", ""68"", ""69"", ""76"", ""80""]","{""level"": 3, ""leading"": 8, ""trailing"": 1}","{""level"": 0, ""leading"": 5, ""trailing"": 1}"
3,Aurélien Tchouaméni_14_RM,31,0.001159,0.001403,0.005889,0.002980,0.000,0.013,6716,80,76,14,0,"[""56"", ""58"", ""62"", ""63"", ""64"", ""69"", ""71"", ""76...","[""54"", ""58"", ""62"", ""63"", ""64"", ""65"", ""66"", ""68...","{""level"": 4, ""leading"": 7, ""trailing"": 3}","{""level"": 8, ""leading"": 2, ""trailing"": 6}"
4,Brahim Díaz_21_RM,31,0.002760,0.000724,0.006109,0.006500,0.500,1.000,2601,34,68,1,0,"[""20"", ""50"", ""61"", ""62"", ""68"", ""69"", ""71"", ""76...","[""25"", ""47"", ""59"", ""60"", ""63"", ""72"", ""73"", ""74...","{""level"": 6, ""leading"": 9, ""trailing"": 4}","{""level"": 5, ""leading"": 12, ""trailing"": 2}"


In [131]:
all_away_players = away_players + away_subs
escaped_players = [player.replace("'", "''") for player in all_away_players]
players_str = ", ".join([f"'{player}'" for player in escaped_players])

try:
    connection = mysql.connector.connect(
        host="localhost",
        user="root",
        password="venomio",
        database="vsp"
    )
    
    if connection.is_connected():       
        cursor = connection.cursor()

        cursor.execute(
            f"""
            SELECT 
                *
            FROM players
            WHERE id IN ({players_str})
            """
            )
        column_names = [desc[0] for desc in cursor.description] 
        away_players_df = pd.DataFrame(cursor.fetchall(), columns=column_names)
            
except Error as e:
    print(f"Error while connecting to MySQL: {e}")
    
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()

away_players_df.head()

,id,team_id,off_xg_coef,def_xg_coef,off_sh_coef,def_sh_coef,fatigue,rhythm,minutes_played,fouls_committed,fouls_drawn,yellow_cards,red_cards,sub_in,sub_out,in_status,out_status
0,Alejandro Balde_3_B,30,0.002458,0.000595,0.006144,0.002266,None,None,6559,55,91,8,0,"[""60"", ""67"", ""68"", ""70"", ""72"", ""82"", ""84""]","[""17"", ""26"", ""46"", ""61"", ""66"", ""70"", ""74"", ""76...","{""level"": 3, ""leading"": 2, ""trailing"": 3}","{""level"": 4, ""leading"": 13, ""trailing"": 1}"
1,Ansu Fati_10_B,30,0.000389,0.000907,0.003016,0.002674,None,None,2269,33,21,5,0,"[""36"", ""59"", ""61"", ""63"", ""64"", ""65"", ""66"", ""70...","[""58"", ""59"", ""62"", ""63"", ""68"", ""69"", ""75"", ""80...","{""level"": 8, ""leading"": 20, ""trailing"": 7}","{""level"": 2, ""leading"": 10, ""trailing"": 0}"
2,Dani Olmo_20_B,30,0.000855,0.000011,-0.000515,0.000181,None,None,607,7,7,0,0,"[""47"", ""48"", ""65""]","[""56"", ""61"", ""75"", ""85"", ""90""]","{""level"": 0, ""leading"": 1, ""trailing"": 2}","{""level"": 1, ""leading"": 4, ""trailing"": 0}"
3,Fermin López_16_B,30,-0.001199,-0.001064,-0.004758,-0.003436,None,None,2425,58,34,4,0,"[""26"", ""37"", ""53"", ""56"", ""63"", ""67"", ""7"", ""70""...","[""46"", ""57"", ""59"", ""61"", ""62"", ""63"", ""64"", ""69...","{""level"": 9, ""leading"": 11, ""trailing"": 4}","{""level"": 4, ""leading"": 9, ""trailing"": 4}"
4,Frenkie de Jong_21_B,30,0.002594,-0.000329,0.007466,0.000139,None,None,7129,80,101,15,0,"[""46"", ""56"", ""59"", ""64"", ""65"", ""75"", ""76"", ""86""]","[""26"", ""36"", ""47"", ""48"", ""57"", ""59"", ""61"", ""70...","{""level"": 2, ""leading"": 7, ""trailing"": 1}","{""level"": 3, ""leading"": 9, ""trailing"": 6}"


In [200]:
home_players_data = _get_players_data(home_players_df, all_home_players)
away_players_data = _get_players_data(away_players_df, all_away_players)

print(home_players_data)
print(away_players_data)

{'Andriy Lunin_13_RM': {'off_sh_coef': -0.00139318, 'def_sh_coef': 0.00628706, 'minutes_played': 4494, 'fouls_committed_rate': 0.00022251891410769915, 'fouls_drawn_rate': 0.0013351134846461949, 'yellow_card_rate': 1.0, 'red_card_rate': 0.0, 'sub_in_count': 0, 'sub_out_count': 0, 'in_status_prob': {'Leading': 0.3333333333333333, 'Level': 0.3333333333333333, 'Trailing': 0.3333333333333333}, 'out_status_prob': {'Leading': 0.3333333333333333, 'Level': 0.3333333333333333, 'Trailing': 0.3333333333333333}, 'fr_xg_off_coef': 0.00114858, 'fr_xg_def_coef': 0.000649297}, 'Éder Militão_3_RM': {'off_sh_coef': 0.000586275, 'def_sh_coef': -0.0036154, 'minutes_played': 7048, 'fouls_committed_rate': 0.01362088535754824, 'fouls_drawn_rate': 0.012627695800227014, 'yellow_card_rate': 0.15625, 'red_card_rate': 0.0, 'sub_in_count': 8, 'sub_out_count': 9, 'in_status_prob': {'Leading': 0.3333333333333333, 'Level': 0.3333333333333333, 'Trailing': 0.3333333333333333}, 'out_status_prob': {'Leading': 0.3333333333

In [117]:
home_avg_subs = round(teams_df[teams_df['team_id'] == home_id].groupby('match_id').size().mean())
away_avg_subs = round(teams_df[teams_df['team_id'] == away_id].groupby('match_id').size().mean())

effective_home_subs = max(0, min(home_avg_subs - (5 - home_subs_avail), home_subs_avail))
effective_away_subs = max(0, min(away_avg_subs - (5 - away_subs_avail), away_subs_avail))

print(home_avg_subs, away_avg_subs, effective_home_subs, effective_away_subs)

4 4 4 4


In [118]:
minutes = minutes_df["sub_minute"].clip(upper=89)
counts = minutes.value_counts().sort_index()
probabilities = counts / counts.sum()

def sample_future_sub_minutes(probabilities, current_minute, n=3):
    future = probabilities[probabilities.index >= current_minute]
    future = future / future.sum()
    
    return np.random.choice(
        future.index,
        size=n,
        replace=False,
        p=future.values
    )

print(sample_future_sub_minutes(probabilities, current_minute, 3))

[70 66 89]


In [151]:
def _get_distribution(team: str, effective_subs: int) -> dict:
    if team == "home":
        team_id = home_id
        avail_subs = home_subs_avail
    else:
        team_id = away_id
        avail_subs = away_subs_avail

    if effective_subs == 0:
        return {100: 0}
    elif effective_subs == 1:
        n_windows = 1
    elif effective_subs < 5:
        n_windows = 2
    else:
        n_windows = 3

    top_minutes = sample_future_sub_minutes(probabilities, current_minute, n_windows)

    base = avail_subs // n_windows
    remainder = avail_subs % n_windows
    distribution = {}
    for i in range(n_windows):
        minute = top_minutes[i]
        distribution[round(min(90, minute))] = distribution.get(minute, 0) + (base + 1 if i < remainder else base)
    return distribution

home_distribution = _get_distribution("home", effective_home_subs)
away_distribution = _get_distribution("away", effective_away_subs)

print(home_distribution)
print(away_distribution)

all_sub_minutes = list(set(list(home_distribution.keys()) + list(away_distribution.keys())))

print(all_sub_minutes)

{84: 3, 73: 2}
{75: 3, 89: 2}
[89, 73, 75, 84]


In [206]:
def _swap_players(active_players: list, inactive_players: list, players_data: dict, subs: int, status: str) -> tuple[list, list, list]:
    if status == 'Trailing':
        off_ratio, def_ratio = 0.8, 0.2
    elif status == 'Leading':
        off_ratio, def_ratio = 0.2, 0.8
    else:
        off_ratio, def_ratio = 0.5, 0.5

    def get_tactical_score(p_name):
        p = players_data[p_name]
        return (p['fr_xg_off_coef'] * off_ratio) + (p['fr_xg_def_coef'] * def_ratio)

    # Sub out
    out_scores = {}
    
    total_minutes_out = sum(players_data[p]['minutes_played'] + 1 for p in active_players)
    total_tactical_out = sum(get_tactical_score(p) + 1 for p in active_players)

    for player in active_players:
        p_data = players_data[player]
        score_history = (p_data['sub_out_count'] + 1) * p_data['out_status_prob'][status] 
        score_hierarchy = 1 / ((p_data['minutes_played'] / 100) + 1)**2
        score_tactical = 1 / (get_tactical_score(player) + 0.1)

        out_scores[player] = score_history * score_hierarchy * score_tactical

    total_out = sum(out_scores.values())
    out_probs = [s / total_out for s in out_scores.values()]
    out_prob_map = {player: (score / total_out) for player, score in out_scores.items()}
    picked_out_players = np.random.choice(active_players, p=out_probs, replace=False, size=subs)

    # Sub in
    in_scores = {}

    total_minutes_in = sum(players_data[p]['minutes_played'] + 1 for p in inactive_players)
    total_tactical_in = sum(get_tactical_score(p) + 1 for p in inactive_players)

    for player in inactive_players:
        p_data = players_data[player]
        score_history = (p_data['sub_in_count'] + 1) * p_data['in_status_prob'][status]
        score_hierarchy = (p_data['minutes_played'] / 100) + 1
        score_tactical = get_tactical_score(player) + 0.1

        in_scores[player] = score_history * score_hierarchy * score_tactical

    total_in = sum(in_scores.values())
    in_probs = [s / total_in for s in in_scores.values()]
    in_prob_map = {player: (score / total_in) for player, score in in_scores.items()}
    picked_in_players = np.random.choice(inactive_players, p=in_probs, replace=False, size=subs)

    active_players = [player for player in active_players if player not in picked_out_players]
    active_players.extend(picked_in_players)
    inactive_players = [player for player in inactive_players if player not in picked_in_players]

    print(status)
    print(f"\n--- Sub Out Probabilities ({status}) ---")
    print(picked_out_players)
    for player, prob in sorted(out_prob_map.items(), key=lambda x: x[1], reverse=True):
        print(f"{player}: {prob:.2%}")

    print(f"\n--- Sub In Probabilities ({status}) ---")
    print(picked_in_players)
    for player, prob in sorted(in_prob_map.items(), key=lambda x: x[1], reverse=True):
        print(f"{player}: {prob:.2%}")
    return active_players, inactive_players, list(picked_in_players)


In [208]:
all_sub_minutes = list(set(list(home_distribution.keys()) + list(away_distribution.keys())))
home_active_players  = home_starters.copy()
away_active_players  = away_players.copy()
home_inactive_players = home_subs.copy()
away_inactive_players = away_subs.copy()

for simulation in range(simulations):
    for minute in range(90):
        if minute in all_sub_minutes:
            home_status = random.choice(["Leading", "Trailing", "Level"])
            away_status = "Leading" if home_status == "Trailing" else "Trailing" if home_status == "Leading" else "Level"
            print(minute)
            if minute in list(home_distribution.keys()):
                home_active_players, home_inactive_players, home_subers = _swap_players(home_active_players, home_inactive_players, home_players_data, home_distribution[minute], home_status)
            if minute in list(away_distribution.keys()):
                away_active_players, away_inactive_players, away_subers = _swap_players(away_active_players, away_inactive_players, away_players_data, away_distribution[minute], away_status)

73
Leading

--- Sub Out Probabilities (Leading) ---
['Kylian Mbappé_9_RM' 'Ferland Mendy_23_RM']
Kylian Mbappé_9_RM: 50.61%
Jude Bellingham_5_RM: 9.77%
Lucas Vázquez_17_RM: 8.92%
Ferland Mendy_23_RM: 7.52%
Eduardo Camavinga_6_RM: 6.58%
Aurélien Tchouaméni_14_RM: 5.18%
Éder Militão_3_RM: 3.44%
Vinicius Júnior_7_RM: 3.31%
Federico Valverde_8_RM: 3.09%
Andriy Lunin_13_RM: 0.82%
Antonio Rüdiger_22_RM: 0.76%

--- Sub In Probabilities (Leading) ---
['Luka Modrić_10_RM' 'Fran Garcia_20_RM']
Luka Modrić_10_RM: 47.37%
Dani Ceballos_19_RM: 25.68%
Brahim Díaz_21_RM: 12.62%
Fran Garcia_20_RM: 10.92%
Arda Güler_15_RM: 2.34%
Endrick_16_RM: 0.60%
Jesús Vallejo_18_RM: 0.44%
Sergio Mestre_34_RM: 0.03%
75
Trailing

--- Sub Out Probabilities (Trailing) ---
['Marc Casado_17_B' 'Fermin López_16_B' 'Pau Cubarsí_2_B']
Fermin López_16_B: 30.61%
Marc Casado_17_B: 25.19%
Pau Cubarsí_2_B: 9.39%
Lamine Yamal_19_B: 9.10%
Pedri_8_B: 7.22%
Raphinha_11_B: 6.86%
Alejandro Balde_3_B: 4.83%
Robert Lewandowski_9_B: 2.38%